# Đồ án cuối kỳ: Phân loại tin nhắn rác (SMS Spam Classification)

- **Môn học:** Nhập môn Học máy
- **Nhóm thực hiện:** 3
- **Bộ dữ liệu:** SMS Spam Collection (5.574 tin nhắn)



In [88]:
import pandas as pd
import numpy as np

df = pd.read_csv('sms+spam+collection/SMSSpamCollection', sep='\t', header=None, names=['Label', 'Text'])

df.head()

,Label,Text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


# CHƯƠNG 2.  TIỀN XỬ LÝ DỮ LIỆU VÀ MÔ HÌNH CƠ SỞ
**Mục tiêu của chương:** 
- Tải, làm sạch và chuẩn hóa dữ liệu văn bản thô. 
- Chuyển đổi ngôn ngữ tự nhiên (chữ) thành các vector toán học (số). 
- Xây dựng và đánh giá nhanh một mô hình học máy cơ bản (Logistic Regression) để làm điểm gốc so sánh (Baseline).

## 2.2 Làm sạch văn bản

In [89]:

import re

df['Label'] = df['Label'].map({'ham':0,'spam':1})

def clean_text(text):
    text=text.lower()
    text = re.sub(r'[^\w\s]','',text)
    return text

df['Clean_Text'] = df['Text'].apply(clean_text)
df[['Label', 'Text', 'Clean_Text']].head() 

,Label,Text,Clean_Text
0,0,"Go until jurong point, crazy.. Available only ...",go until jurong point crazy available only in ...
1,0,Ok lar... Joking wif u oni...,ok lar joking wif u oni
2,1,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in 2 a wkly comp to win fa cup fina...
3,0,U dun say so early hor... U c already then say...,u dun say so early hor u c already then say
4,0,"Nah I don't think he goes to usf, he lives aro...",nah i dont think he goes to usf he lives aroun...


## 2.3 Mã hóa từ vựng (Tokenization)

In [90]:
from tensorflow.keras.preprocessing.text import Tokenizer

max_words = 10000 
tokenizer = Tokenizer(num_words = max_words) #chỉ học và nhớ 10000 từ xuất hiện nhiều nhất

tokenizer.fit_on_texts(df['Clean_Text'])
sequences = tokenizer.texts_to_sequences(df['Clean_Text'])


## 2.4 Đệm chuỗi (Padding) và Phân chia dữ liệu (Train - Test Split)

In [91]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

# Chèn số 0 (Padding) - ép tất cả tin nhắn về cùng chiều dài 100 từ
max_len = 100
X_padded = pad_sequences(sequences, maxlen=max_len, padding='post', truncating='post')

# Chia dữ liệu: 80% để Học (Train), 20% để Thi (Test)
Y = df['Label'].values
X_train, X_test, Y_train, Y_test = train_test_split(X_padded, Y, test_size=0.2, random_state=42)

print('Số tin nhắn để HỌC (Train):', len(X_train))
print('Số tin nhắn để THI (Test):', len(X_test))


Số tin nhắn để HỌC (Train): 4457
Số tin nhắn để THI (Test): 1115


## 2.5 Mô hình cơ sở (Logistic Regression)
Mô hình truyền thống được xây dựng làm điểm gốc (Baseline) để so sánh hiệu suất với các mô hình Deep Learning.


In [92]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Logistic Regression
baseline_model = LogisticRegression(max_iter=1000)
baseline_model.fit(X_train, Y_train)

# Đánh giá
Y_pred = baseline_model.predict(X_test)
print("KẾT QUẢ MÔ HÌNH CƠ SỞ (Logistic Regression)")
print(f"Accuracy: {accuracy_score(Y_test, Y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(Y_test, Y_pred, target_names=['Ham', 'Spam']))


KẾT QUẢ MÔ HÌNH CƠ SỞ (Logistic Regression)
Accuracy: 0.8735

Classification Report:
              precision    recall  f1-score   support

         Ham       0.89      0.97      0.93       966
        Spam       0.56      0.23      0.33       149

    accuracy                           0.87      1115
   macro avg       0.73      0.60      0.63      1115
weighted avg       0.85      0.87      0.85      1115

